# Universe Coverage (Slices 6A–6C)

OwnerLens's SEC-first canonical pipeline was validated on a handful of companies. Before designing the next normalization improvement, this notebook measures how it holds up across **24 conventional U.S. operating companies** in several industries:

```
SEC fetch → canonical history → owner economics → capital efficiency → economic value
          → compounding → capital allocation → summary → coverage
```

This is **coverage discovery**. Nothing here changes SEC normalization or adds ticker overrides. Every blocker is diagnosed from the raw Company Facts using a small read-only catalog of alternative SEC concepts.

The survey was run with `uv run python scripts/survey_universe.py`, which ingests through the existing `ingest_company` into a dedicated `data/universe_6a.db`. This notebook replays it **offline** from the stored raw snapshots, so it is reproducible and makes no SEC requests.

In [1]:
import json
from pathlib import Path

from owner_lens.persistence import FilesystemRawSnapshotStore
from owner_lens.sec_adapter import canonical_history_from_sec
from owner_lens.universe import (
    UNIVERSE_6A,
    UniverseReport,
    format_company_view,
    format_metric_view,
    format_pattern_view,
    survey_from_snapshot,
)
from owner_lens.universe import DiagnosisCategory as D

repo_root = next((p for p in (Path('.'), Path('..')) if (p / 'pyproject.toml').exists()), Path('.'))
manifest = json.loads((repo_root / 'data' / 'universe_6a_report.json').read_text(encoding='utf-8'))
raw_store = FilesystemRawSnapshotStore(repo_root / 'data' / 'raw' / 'sec' / 'company_facts')
report = UniverseReport(tuple(survey_from_snapshot(entry, raw_store) for entry in manifest['companies']))
print(f'{len(report.companies)} companies replayed offline from stored SEC snapshots')
for member in UNIVERSE_6A:
    if member.note:
        print(f'  {member.ticker}: {member.note}')

24 companies replayed offline from stored SEC snapshots
  CAT: stress test: captive finance arm (Cat Financial)
  DE: stress test: captive finance arm (John Deere Financial)
  UNH: stress test: insurance-like balance sheet


## 1. Company view

Each row shows the six analytical layers (`OK` available, `PART` partial, `INSUF` insufficient data, `UNAV` unsupported input, `BLOCK` invalid input, `ERR` unexpected exception) and the first blocking metric with its diagnosis.

`FULL` requires every layer to be available **and** no silently stale metric.

In [2]:
print(format_company_view(report))

Ticker Overall  Owner CapEf EcVal Comp  CapAl Summ   Unsupported                        Invalid                  Stale (silent)               Primary blocker
ADBE   FULL     OK    OK    OK    OK    OK    OK     -                                  -                        -                            -
V      PARTIAL  PART  OK    INSUF INSUF PART  INSUF  diluted_shares                     -                        -                            diluted_shares [MISSING_CONCEPT]
COST   FULL     OK    OK    OK    OK    OK    OK     -                                  -                        -                            -
MSFT   FULL     OK    OK    OK    OK    OK    OK     -                                  -                        -                            -
CRM    PARTIAL  OK    BLOCK BLOCK BLOCK BLOCK BLOCK  -                                  long_term_debt           -                            long_term_debt [AMBIGUOUS_DUPLICATE]
NOW    PARTIAL  OK    UNAV  UNAV  UNAV  UNAV  UNAV   cur

## 2. Metric view: which canonical metrics block coverage?

For each canonical metric: how many companies have it available, unsupported, invalid (ambiguous or malformed), or structurally absent; how many companies it blocks; and the dominant diagnosis.

In [3]:
print(format_metric_view(report))

Metric                    Avail Unsup Inval Absent Blocks  Dominant diagnosis   Companies affected
revenue                      22     1     1      0      2  AMBIGUOUS_DUPLICATE  unsupported: XOM; invalid: PFE
operating_income             19     5     0      0      5  MISSING_CONCEPT      unsupported: NKE,DE,PFE,XOM,CVX
net_income                   22     2     0      0      1  STALE_CONCEPT        unsupported: CAT,XOM
operating_cash_flow          23     1     0      0      0  PERIOD_ISSUE         unsupported: XOM
capital_expenditures         22     1     1      0      1  AMBIGUOUS_DUPLICATE  unsupported: XOM; invalid: META
diluted_shares               22     2     0      0      1  MISSING_CONCEPT      unsupported: V,XOM
income_tax_expense           23     1     0      0      0  PERIOD_ISSUE         unsupported: XOM
pretax_income                22     2     0      0      1  STALE_CONCEPT        unsupported: ORCL,XOM
repurchases                  21     3     0      0      1  MISSING_CON

## 3. Patterns: recurring taxonomy versus company quirks

In [4]:
print(format_pattern_view(report))

Top blocking metrics (companies blocked):
  operating_income          5
  current_debt              4
  long_term_debt            3
  cash                      2
  revenue                   2
Diagnosis categories (company-metric findings):
  ALTERNATIVE_CONCEPT    4
  AMBIGUOUS_DUPLICATE    5
  BUSINESS_STRUCTURE     1
  COMPOSITION_BLOCKED    3
  MISSING_CONCEPT        14
  PERIOD_ISSUE           13
  STALE_CONCEPT          14
Silently stale AVAILABLE metrics (selected concept has no recent value): 0 in 0 companies
Ambiguous duplicates by restatement kind:
  VALUE_CHANGE    5  CRM:long_term_debt, META:capital_expenditures, INTU:long_term_debt, PFE:revenue, PFE:cash
Conflicts resolved by the normalizer (precision / stock split): 15
  NOW    net_income             FY2021 PRECISION
  NOW    diluted_shares         FY2024 STOCK_SPLIT
  NOW    diluted_shares         FY2023 STOCK_SPLIT
  NOW    diluted_shares         FY2022 STOCK_SPLIT x5
  NOW    diluted_shares         FY2021 STOCK_SPLIT x5

## 4. The most important finding: silent staleness (fixed in Slice 6B)

In the 6A survey the SEC selector took the first preferred concept with *any* annual history. When a company stopped tagging that concept, OwnerLens kept the old values and marked the metric `AVAILABLE`. 14 metrics in 10 companies were affected, for example NVIDIA revenue (ending FY2022), Amazon capital expenditures (ending FY2016), and Caterpillar net income (ending FY2010).

Slice 6B made concept selection recency-aware. A concept is eligible only if it covers the company's latest fiscal year, so stale history can no longer appear as `AVAILABLE`. A current fallback concept is selected when one exists in the preference list; otherwise the metric is `UNSUPPORTED` and diagnosed `STALE_CONCEPT`. The cell below lists any silently stale metric. After 6B it should print nothing.

In [5]:
for company in report.surveyed:
    for metric in company.silently_stale():
        print(f'{company.ticker:<5} {metric:<24} {company.metrics[metric].reason[:120]}')

## 4b. Restatement and stock-split conflicts (resolved in Slice 6C)

The same fiscal year often has several reported values because later 10-Ks restate comparatives. 6C resolves only the mechanical cases. **Precision re-roundings** are an exact rounding to thousands through millions, never a percentage band. **Stock splits** apply to share counts only, need an integer forward ratio and a consistent filing order, and older pre-split-only years are scaled to the post-split basis. Genuine revisions still fail loudly. Every resolved value keeps its as-filed value, the split factor, and the superseded reports.

In [6]:
print(f'Resolved conflicts: {len(report.resolved_conflicts())}')
for ticker, metric, entry in report.resolved_conflicts():
    print(f'  {ticker:<5} {metric:<22} {entry}')
print()
print('Still failing loudly (genuine revisions):')
for kind, items in report.restatements().items():
    print(f'  {kind:<13} {", ".join(items)}')

Resolved conflicts: 15
  NOW   net_income             FY2021 PRECISION
  NOW   diluted_shares         FY2024 STOCK_SPLIT
  NOW   diluted_shares         FY2023 STOCK_SPLIT
  NOW   diluted_shares         FY2022 STOCK_SPLIT x5
  NOW   diluted_shares         FY2021 STOCK_SPLIT x5
  NOW   cash                   FY2020 PRECISION
  NOW   total_assets           FY2020 PRECISION
  NOW   total_equity           FY2020 PRECISION
  AMZN  diluted_shares         FY2021 STOCK_SPLIT
  NVDA  diluted_shares         FY2024 STOCK_SPLIT
  NVDA  diluted_shares         FY2023 STOCK_SPLIT
  NVDA  diluted_shares         FY2022 STOCK_SPLIT x10
  CMG   diluted_shares         FY2023 STOCK_SPLIT
  CMG   diluted_shares         FY2022 STOCK_SPLIT
  CMG   diluted_shares         FY2021 STOCK_SPLIT x50

Still failing loudly (genuine revisions):
  VALUE_CHANGE  CRM:long_term_debt, META:capital_expenditures, INTU:long_term_debt, PFE:revenue, PFE:cash


In [7]:
nvda = next(entry for entry in manifest['companies'] if entry['ticker'] == 'NVDA')
history = canonical_history_from_sec(json.loads(raw_store.get(nvda['content_hash'])), ticker='NVDA')
for fact in history.series_for('diluted_shares').observations:
    r = fact.resolution
    detail = '' if r is None else (
        f'  {r.kind.value} x{r.split_factor}, as filed {r.reported_value:,}; superseded '
        + ', '.join(f'{s.value:,} ({s.filed})' for s in r.superseded)
    )
    print(f'FY{fact.fiscal_year} {fact.value:>16,}  {fact.accession}{detail}')

FY2026   24,514,000,000  0001045810-26-000021
FY2025   24,804,000,000  0001045810-25-000023
FY2024   24,940,000,000  0001045810-25-000023  STOCK_SPLIT x1, as filed 24,940,000,000; superseded 2,494,000,000 (2024-02-21)
FY2023   25,070,000,000  0001045810-25-000023  STOCK_SPLIT x1, as filed 25,070,000,000; superseded 2,507,000,000 (2023-02-24), 2,507,000,000 (2024-02-21)
FY2022   25,350,000,000  0001045810-22-000036  STOCK_SPLIT x10, as filed 2,535,000,000; superseded 


## 4c. Current debt: composed from its components (Slice 6D)

Current debt was the largest recurring blocker (9 of 24 companies). SEC filers split it three ways, and 6D resolves all three through registry policy with no per-company override:

* a **total** concept (`DebtCurrent`) wins when reported, so components are never double counted;
* otherwise the **components** reported at each year end are added (`LongTermDebtCurrent`, `CommercialPaper`, `ShortTermBorrowings`, `OtherShortTermBorrowings`);
* otherwise the company's own debt totals may **prove** there is nothing to report.

Lease liabilities are never summed, and `LongTermDebtAndCapitalLeaseObligationsCurrent` bundles debt with leases so it refuses composition rather than understating (HD, KO, LOW).

In [8]:
for ticker in ('MSFT', 'AMZN', 'COST', 'META', 'HD'):
    entry = next(e for e in manifest['companies'] if e['ticker'] == ticker)
    series = canonical_history_from_sec(
        json.loads(raw_store.get(entry['content_hash'])), ticker=ticker
    ).series_for('current_debt')
    print(f'{ticker}  {series.status.value}')
    for fact in series.observations:
        parts = '' if not fact.components else '   = ' + ' + '.join(
            f'{c.provider_field} {c.value / 1e6:,.0f}M' for c in fact.components)
        print(f'   FY{fact.fiscal_year} {fact.value / 1e6:>10,.0f}M  {fact.provider_field.split(" + ")[0]}{parts}')
    if not series.observations:
        print(f'   {(series.reason or "")[:150]}')

MSFT  AVAILABLE


   FY2026      9,227M  LongTermDebtCurrent
   FY2025      2,999M  LongTermDebtCurrent
   FY2024      8,942M  LongTermDebtCurrent   = LongTermDebtCurrent 2,249M + CommercialPaper 6,693M
   FY2023      5,247M  LongTermDebtCurrent
   FY2022      2,749M  LongTermDebtCurrent
   FY2021      8,072M  LongTermDebtCurrent
AMZN  AVAILABLE
   FY2025      3,203M  LongTermDebtCurrent   = LongTermDebtCurrent 2,748M + ShortTermBorrowings 455M
   FY2024      5,168M  LongTermDebtCurrent   = LongTermDebtCurrent 5,017M + ShortTermBorrowings 151M
   FY2023      8,641M  LongTermDebtCurrent   = LongTermDebtCurrent 8,494M + ShortTermBorrowings 147M
   FY2022      2,999M  LongTermDebtCurrent
   FY2021      1,491M  LongTermDebtCurrent
   FY2020      1,155M  LongTermDebtCurrent


COST  AVAILABLE
   FY2025         75M  LongTermDebtCurrent
   FY2024        103M  LongTermDebtCurrent
   FY2023      1,081M  LongTermDebtCurrent
   FY2022        161M  LongTermDebtCurrent   = LongTermDebtCurrent 73M + OtherShortTermBorrowings 88M
   FY2021        840M  LongTermDebtCurrent   = LongTermDebtCurrent 799M + OtherShortTermBorrowings 41M
   FY2020         95M  LongTermDebtCurrent
META  STRUCTURALLY_ABSENT
   current_debt is structurally absent: the company reports LongTermDebt equal to LongTermDebtNoncurrent (or zero) at its latest fiscal-year end, so ther
HD  UNSUPPORTED
   LongTermDebtAndCapitalLeaseObligationsCurrent is non-zero at 2021-01-31, 2022-01-30, 2023-01-29, 2024-01-28, 2025-02-02, 2026-02-01 and bundles econom


In [9]:
from collections import Counter

blocked = [c.ticker for c in report.surveyed
           if any(l.blocking_metric == 'current_debt' for l in c.layers.values())]
statuses = {c.ticker: c.metrics['current_debt'].status.value for c in report.surveyed if c.metrics}
print(f'companies blocked by current debt: {len(blocked)} {blocked}')
print('current-debt statuses:', dict(Counter(statuses.values())))
print('structurally absent (proven):', [t for t, s in statuses.items() if s == 'STRUCTURALLY_ABSENT'])

companies blocked by current debt: 4 ['NOW', 'KO', 'HD', 'LOW']
current-debt statuses: {'AVAILABLE': 16, 'UNSUPPORTED': 6, 'STRUCTURALLY_ABSENT': 2}
structurally absent (proven): ['META', 'CMG']


## 5. What would the next normalization improvement unblock?

`unblocked_by` is a what-if over the diagnoses. It counts companies whose every unsupported or invalid required input would be resolved by an improvement. It does not recompute anything, and it ignores silent staleness, so it measures blocked coverage, not trustworthiness.

In [10]:
catalog = lambda t, f: f.diagnosis in (D.ALTERNATIVE_CONCEPT, D.STALE_CONCEPT) and bool(f.candidates)
long_term_debt = lambda t, f: f.metric == 'long_term_debt'
revisions = lambda t, f: f.diagnosis is D.AMBIGUOUS_DUPLICATE
current_debt = lambda t, f: f.metric == 'current_debt'

scenarios = {
    'Catalog concept policy (restricted cash, NCI equity, ProfitLoss)': catalog,
    'Long-term-debt alternatives and restatements': long_term_debt,
    'A policy for genuine revisions': revisions,
    'Remaining current debt (debt/lease split)': current_debt,
    'Catalog + long-term debt': lambda t, f: catalog(t, f) or long_term_debt(t, f),
    'Catalog + long-term debt + revisions': lambda t, f: catalog(t, f) or long_term_debt(t, f) or revisions(t, f),
}
blocked = [c.ticker for c in report.surveyed if c.unusable_inputs()]
print(f'{len(blocked)} of {len(report.companies)} companies have at least one blocking input\n')
for name, resolves in scenarios.items():
    unblocked = report.unblocked_by(resolves)
    print(f'{name:<62} unblocks {len(unblocked):>2}: {", ".join(unblocked)}')

18 of 24 companies have at least one blocking input

Catalog concept policy (restricted cash, NCI equity, ProfitLoss) unblocks  3: PG, CAT, UNH
Long-term-debt alternatives and restatements                   unblocks  2: CRM, INTU
A policy for genuine revisions                                 unblocks  3: CRM, META, INTU
Remaining current debt (debt/lease split)                      unblocks  2: HD, LOW
Catalog + long-term debt                                       unblocks  5: CRM, INTU, PG, CAT, UNH
Catalog + long-term debt + revisions                           unblocks  6: CRM, META, INTU, PG, CAT, UNH


## 6. Acceptance answers

In [11]:
total = len(report.companies)
dist = report.distribution()
print('1-2. Coverage:', ', '.join(f'{k} {v}/{total} ({v / total:.0%})' for k, v in dist.items()))
print('3.   Top blocking metrics:', ', '.join(f'{m} ({n})' for m, n in report.top_blockers()))
recurring = [r for r in report.recurring_candidates() if r[3]]
print(f'4.   Recurring alternative concepts (3+ companies): {len(recurring)};',
      'quirks:', len(report.recurring_candidates()) - len(recurring))
composites = report.composites()
print(f'5.   Composite candidates: {len(composites)} in {len({c[0] for c in composites})} companies',
      f'({sum(1 for c in composites if c[2])} value-level understatements)')
stale = [c for c in report.surveyed if c.silently_stale()]
print(f'     Silently stale metrics: {sum(len(c.silently_stale()) for c in stale)} in {len(stale)} companies')
print('     Diagnosis mix:', report.diagnosis_counts())

1-2. Coverage: FULL 5/24 (21%), PARTIAL 12/24 (50%), FAILED 7/24 (29%)
3.   Top blocking metrics: operating_income (5), current_debt (4), long_term_debt (3), cash (2), revenue (2)
4.   Recurring alternative concepts (3+ companies): 4; quirks: 4
5.   Composite candidates: 0 in 0 companies (0 value-level understatements)
     Silently stale metrics: 0 in 0 companies
     Diagnosis mix: {'ALTERNATIVE_CONCEPT': 4, 'AMBIGUOUS_DUPLICATE': 5, 'BUSINESS_STRUCTURE': 1, 'COMPOSITION_BLOCKED': 3, 'MISSING_CONCEPT': 14, 'PERIOD_ISSUE': 13, 'STALE_CONCEPT': 14}


**Reading the answers**

The prose below records the original **6A** survey. The cells above replay the stored snapshots with the *current* code, so they show the results after Slice 6B (recency-aware selection), 6C (conflict resolution), and 6D (current-debt composition). The before and after comparisons are in the Feature 6 documentation.

* **Coverage:** 6A reached 3 of 24 full (12%), 12 partial, 9 failed. After 6B: 3 full, 10 partial, 11 failed (CAT and DE began failing honestly on stale concepts). After 6C: 4 full, 13 partial, 7 failed. After 6D: **5 full (adds CMG), 12 partial, 7 failed**.
* **Top blockers:** current debt dominated every earlier survey. After 6D it is resolved for 16 of 24 companies and proven absent for 2 more, so **operating income** is now the top blocker (5 companies), ahead of the remaining current-debt cases (4) and long-term debt (3).
* **Recurring versus quirk:** failures are **mostly recurring taxonomy patterns**, which is why general rules kept working: recency (6B), mechanical restatements (6C), and additive components (6D) each cleared a whole class without a single ticker override.
* **Composite metrics:** current debt was the only metric needing a sum, in 5 companies. 6D implements it, so the survey's composite-candidate list is now empty.
* **SEC-first at 100+:** practical. Three consecutive slices of general rules moved coverage without per-company conditionals, and the override registry shrank (6D removed the current-debt overrides for V, COST, and MSFT).
* **Next improvement:** a catalog-concept policy (restricted cash, noncontrolling-interest equity, `ProfitLoss`), then long-term-debt alternatives. Operating-income derivation, the debt/lease split (HD, KO, LOW), and genuine revisions remain separate decisions.